# NB09A — CAMELYON16 Metastasis Detection (5-Fold CV)

Binary classification of lymph node metastasis on the 269 labeled CAMELYON16 training slides (after quality filtering); the 130 official test slides remain held out. 5-fold stratified cross-validation with LogisticRegression(C=1.0, L2). Reports AUROC, accuracy, F1, average precision, with bootstrap 95% CI from 1000 replicates.

Saves `cam16_oof.csv` (slide-level OOF probabilities) for NB13 figure 4A ROC rendering.

In [ ]:
import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, average_precision_score

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
OUTDIR    = WORKSPACE / 'results' / 'cam16_eval'
EMB_INDEX = WORKSPACE / 'embeddings' / 'camelyon16_index.csv'
OUTDIR.mkdir(parents=True, exist_ok=True)

print('CAMELYON16 binary metastasis detection (5-fold CV)')
if not EMB_INDEX.exists():
    print(f'[SKIP] {EMB_INDEX} not found; run NB08 first with CAMELYON16 data')
    raise SystemExit(0)

df_idx = pd.read_csv(EMB_INDEX)
X = []; slide_ids = []
for _, row in df_idx.iterrows():
    try:
        v = np.load(row['path_emb']).astype(np.float32)
        if v.ndim == 1 and v.shape[0] == 768:
            X.append(v); slide_ids.append(row['slide_id'])
    except Exception:
        pass
X = np.stack(X, axis=0) if X else np.zeros((0, 768), dtype=np.float32)
y = np.array([1 if 'tumor' in str(sid).lower() else 0 for sid in slide_ids], dtype=int)
print(f'[DATA] {len(X)} slides; {y.sum()} tumor; {(1 - y).sum()} normal')

if len(X) < 10:
    print('[SKIP] too few slides'); raise SystemExit(0)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(y), dtype=np.float64)
oof_preds = np.zeros(len(y), dtype=int)

for fold, (tr, te) in enumerate(skf.split(X, y), 1):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X[tr])
    X_te = scaler.transform(X[te])
    clf = LogisticRegression(C=1.0, penalty='l2', max_iter=5000, random_state=42)
    clf.fit(X_tr, y[tr])
    oof_probs[te] = clf.predict_proba(X_te)[:, 1]
    oof_preds[te] = clf.predict(X_te)
    fold_auc = roc_auc_score(y[te], oof_probs[te])
    print(f'  fold {fold}: AUROC={fold_auc:.3f} (n_test={len(te)})')

auroc = roc_auc_score(y, oof_probs)
acc = accuracy_score(y, oof_preds)
f1 = f1_score(y, oof_preds)
avg_prec = average_precision_score(y, oof_probs)

rng = np.random.default_rng(42)
auc_boots = []
for _ in range(1000):
    idx = rng.choice(len(y), size=len(y), replace=True)
    try:
        auc_boots.append(roc_auc_score(y[idx], oof_probs[idx]))
    except Exception:
        pass
ci_lo = float(np.percentile(auc_boots, 2.5))
ci_hi = float(np.percentile(auc_boots, 97.5))

results = {
    'auroc': float(auroc), 'auroc_ci95': [ci_lo, ci_hi],
    'accuracy': float(acc), 'f1': float(f1), 'avg_precision': float(avg_prec),
    'n_slides': int(len(y)), 'n_tumor': int(y.sum()), 'n_normal': int((1 - y).sum()),
}
(OUTDIR / 'cam16_results.json').write_text(json.dumps(results, indent=2))

pd.DataFrame({
    'slide_id': slide_ids, 'y_true': y,
    'y_pred': oof_preds, 'p_tumor': oof_probs,
}).to_csv(OUTDIR / 'cam16_oof.csv', index=False)

print(f'\n  AUROC: {auroc:.3f}  95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'  accuracy: {acc:.3f}  F1: {f1:.3f}  avg precision: {avg_prec:.3f}')
print(f'  results saved to: {OUTDIR}')
print('NB09A complete. Next: NB10 (PANDA feature processing).')